In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors opencv-python-headless datasets

In [ ]:
import os, math, json, random, copy, re, time, pickle
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageDraw, ImageOps
import cv2, matplotlib.pyplot as plt

assert torch.cuda.is_available(), "Enable a GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1e9))
device = "cuda"; DT = torch.float16
torch.backends.cudnn.benchmark = True

RES          = 512          
T_TOTAL      = 1000
TOP_FRAC     = 0.25         
DATA_SOURCE  = "coco"      
N_DATA       = 3000         
N_TEST       = 24           
TRAIN_STEPS  = 3000         
ACCUM        = 4
LR, ENC_LR   = 2e-4, 2e-5   
LAMBDAS      = (1.0, 2.0, 0.05)  
MAX_MINUTES  = 100          
CACHE_DIR    = "/content/cache"
try:                                  
    from google.colab import drive; drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/sketchingreality"
except Exception as e: print("Drive not mounted, using local cache (will be lost on disconnect):", str(e)[:80])
os.makedirs(CACHE_DIR, exist_ok=True)
CKPT = f"{CACHE_DIR}/ckpt.pt"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(0)

def show(imgs, titles=None, size=3.2):
    fig, axs = plt.subplots(1, len(imgs), figsize=(size*len(imgs), size)); axs = np.atleast_1d(axs)
    for i, (a, im) in enumerate(zip(axs, imgs)):
        a.imshow(im, cmap="gray" if (hasattr(im, "ndim") and im.ndim == 2) else None); a.axis("off")
        if titles: a.set_title(titles[i], fontsize=8)
    plt.tight_layout(); plt.show()

def load_user_sketch(path, size=RES):
    """Open a png/jpg (transparent backgrounds become white) -> RGB PIL, white bg, dark strokes."""
    im = Image.open(path).convert("RGBA"); bg = Image.new("RGBA", im.size, "white"); bg.alpha_composite(im)
    return bg.convert("RGB").resize((size, size))

def make_demo_sketch(seed=1, size=RES):
    rng = np.random.RandomState(seed); im = Image.new("RGB", (size, size), "white"); d = ImageDraw.Draw(im); s = size/512
    wob = lambda pts, j=3: [(x + rng.randn()*j, y + rng.randn()*j) for x, y in pts]
    line = lambda pts, w=3, j=3: d.line(wob(pts, j), fill="black", width=w, joint="curve")
    line([(0,330*s),(120*s,290*s),(260*s,300*s),(400*s,330*s),(512*s,320*s)])
    line([(150*s,420*s),(160*s,300*s),(150*s,240*s)]); line([(215*s,420*s),(205*s,300*s),(215*s,240*s)])
    for k in range(5):
        cx, cy, r = 185*s + rng.randn()*8, 150*s + rng.randn()*6, 90*s + rng.randn()*5
        a = np.linspace(0, 2*np.pi, 30); line([(cx + r*np.cos(t), cy + 0.8*r*np.sin(t)) for t in a], w=2, j=4)
    line([(300*s,400*s),(450*s,400*s),(450*s,420*s),(300*s,420*s),(300*s,400*s)])
    line([(300*s,360*s),(450*s,360*s)]); line([(300*s,380*s),(450*s,380*s)])
    line([(305*s,420*s),(305*s,445*s)]); line([(445*s,420*s),(445*s,445*s)])
    return im
demo_sketch = make_demo_sketch(); DEMO_PROMPT = "a big tree and a bench near the hillside"

In [ ]:
class DoubleConv(nn.Module):
    def __init__(s, cin, cout):
        super().__init__(); gn = lambda c: nn.GroupNorm(min(8, c), c)
        s.net = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), gn(cout), nn.SiLU(),
                              nn.Conv2d(cout, cout, 3, padding=1), gn(cout), nn.SiLU())
    def forward(s, x): return s.net(x)

def timestep_embedding(t, dim=128):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float()[:, None] * freqs[None]; return torch.cat([a.cos(), a.sin()], -1)

class ModulationNet(nn.Module):
    def __init__(s, sketch_ch=512, lat_ch=4):
        super().__init__()
        s.sketch_path = nn.Sequential(DoubleConv(sketch_ch, 256), DoubleConv(256, 128))
        mk = lambda: nn.ModuleList([DoubleConv(lat_ch, 16), DoubleConv(16, 32), DoubleConv(32, 64), DoubleConv(64, 128)])
        s.eps_path, s.lat_path = mk(), mk()
        s.t_mlp = nn.Sequential(nn.Linear(128, 256), nn.SiLU(), nn.Linear(256, 256))
        s.fuse = DoubleConv(384, 256)
        s.up1, s.uc1 = nn.ConvTranspose2d(256, 128, 2, 2), DoubleConv(128, 64)
        s.up2, s.uc2 = nn.ConvTranspose2d(64, 32, 2, 2),  DoubleConv(32, 16)
        s.up3, s.uc3 = nn.ConvTranspose2d(16, 8, 2, 2),   DoubleConv(8, 8)
        s.final = nn.Conv2d(8, 2 * lat_ch, 1)
        nn.init.normal_(s.final.weight, std=1e-2); nn.init.zeros_(s.final.bias)
    @staticmethod
    def _down(path, x):
        for i, blk in enumerate(path): x = blk(x if i == 0 else F.max_pool2d(x, 2))
        return x
    def forward(s, eps, zt, sk, t):
        p = s._down(s.eps_path, eps); l = s._down(s.lat_path, zt)
        k = F.interpolate(s.sketch_path(sk), size=p.shape[-2:], mode="bilinear", align_corners=False)
        h = s.fuse(torch.cat([k, p, l], 1)) + s.t_mlp(timestep_embedding(t))[:, :, None, None]
        h = s.uc1(s.up1(h)); h = s.uc2(s.up2(h)); h = s.uc3(s.up3(h))
        S, B = s.final(h).chunk(2, dim=1)
        return eps * (1 + S) + B, S, B

In [ ]:
from transformers import CLIPModel, CLIPTokenizer
CLIP_ID = "openai/clip-vit-base-patch16"
CLIP_MEAN = torch.tensor([0.4815, 0.4578, 0.4082]).view(3,1,1); CLIP_STD = torch.tensor([0.2686, 0.2613, 0.2758]).view(3,1,1)

def _norm_pil(pil):
    im = pil.convert("RGB").resize((224, 224), Image.BICUBIC)
    x = torch.from_numpy(np.asarray(im).copy()).float().div(255).permute(2, 0, 1)
    return (x - CLIP_MEAN) / CLIP_STD
sketch_to_tensor = _norm_pil
def sketch_np_to_tensor(a): return _norm_pil(Image.fromarray(a))

class SketchEncoder(nn.Module):
    def __init__(s, ft_layers=3):
        super().__init__()
        s.clip = CLIPModel.from_pretrained(CLIP_ID).float()
        s.frozen = CLIPModel.from_pretrained(CLIP_ID).float().eval()
        s.tok = CLIPTokenizer.from_pretrained(CLIP_ID)
        for p in s.parameters(): p.requires_grad_(False)
        if ft_layers > 0:
            for layer in s.clip.vision_model.encoder.layers[-ft_layers:]:
                for p in layer.parameters(): p.requires_grad_(True)
            for m in (s.clip.vision_model.post_layernorm, s.clip.visual_projection):
                for p in m.parameters(): p.requires_grad_(True)
    def forward(s, x):
        vm = s.clip.vision_model; h = vm(pixel_values=x).last_hidden_state
        f = s.clip.visual_projection(vm.post_layernorm(h[:, 1:]))
        return f.transpose(1, 2).reshape(x.shape[0], 512, 14, 14)
    @torch.no_grad()
    def dense_feats(s, x):
        vm = s.frozen.vision_model; hs = vm(pixel_values=x, output_hidden_states=True).hidden_states[-2]
        last = vm.encoder.layers[-1]
        v = last.self_attn.out_proj(last.self_attn.v_proj(last.layer_norm1(hs)))
        return F.normalize(s.frozen.visual_projection(vm.post_layernorm(v))[:, 1:], dim=-1)
    @torch.no_grad()
    def sketch_embed(s, x): return F.normalize(s.dense_feats(x).mean(1), dim=-1)           
    @torch.no_grad()
    def image_embed(s, x):
        pooled = s.frozen.vision_model(pixel_values=x).pooler_output
        return F.normalize(s.frozen.visual_projection(pooled), dim=-1)

enc = SketchEncoder(ft_layers=3).to(device)
mod = ModulationNet().to(device)
mod_ema = copy.deepcopy(mod).eval().requires_grad_(False)
print("trainable: modulation %.1fM | encoder %.1fM" % (sum(p.numel() for p in mod.parameters())/1e6,
      sum(p.numel() for p in enc.parameters() if p.requires_grad)/1e6))

In [ ]:
from diffusers import StableDiffusionPipeline, DDIMScheduler, DPMSolverMultistepScheduler
from diffusers.models.attention_processor import AttnProcessor2_0

pipe = None
for rid in ["stabilityai/stable-diffusion-2-1-base", "sd2-community/stable-diffusion-2-1-base"]:
    try: pipe = StableDiffusionPipeline.from_pretrained(rid, torch_dtype=DT, safety_checker=None).to(device); break
    except Exception as e: print("could not load", rid, "->", str(e)[:80])
assert pipe is not None, "Could not load SD2.1-base"
ddim = DDIMScheduler.from_config(pipe.scheduler.config)
dpm  = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.scheduler = ddim
unet, vae = pipe.unet.eval(), pipe.vae.eval()
for m in (unet, vae, pipe.text_encoder): m.requires_grad_(False)
SCALE = vae.config.scaling_factor; ALPHAS = ddim.alphas_cumprod.to(device)
LAT = RES // 8; MASK_RES = LAT // 2
ATTN_N = {(LAT // 8)**2, (LAT // 4)**2, (LAT // 2)**2}
MASK_LAYERS = sorted(int(math.sqrt(n)) for n in ATTN_N)[1:]       

class AttnStore:
    def __init__(s): s.active, s.maps = False, []
class StoreProc:
    def __init__(s, store): s.store, s.default = store, AttnProcessor2_0()
    def __call__(s, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None, **kw):
        if encoder_hidden_states is None or not s.store.active:
            return s.default(attn, hidden_states, encoder_hidden_states, attention_mask, temb)
        B, N, _ = hidden_states.shape
        q = attn.head_to_batch_dim(attn.to_q(hidden_states)); k = attn.head_to_batch_dim(attn.to_k(encoder_hidden_states))
        v = attn.head_to_batch_dim(attn.to_v(encoder_hidden_states))
        probs = attn.get_attention_scores(q, k, None)
        if N in ATTN_N: s.store.maps.append(probs.view(B, attn.heads, N, -1).mean(1).float())
        out = attn.batch_to_head_dim(torch.bmm(probs, v))
        return attn.to_out[1](attn.to_out[0](out))
store = AttnStore(); unet.set_attn_processor(StoreProc(store))

_EMB = {}
@torch.no_grad()
def get_emb(prompt):
    if prompt not in _EMB:
        ids = pipe.tokenizer(prompt, padding="max_length", max_length=pipe.tokenizer.model_max_length,
                             truncation=True, return_tensors="pt").input_ids.to(device)
        _EMB[prompt] = pipe.text_encoder(ids)[0]
    return _EMB[prompt]

def find_token_idx(prompt, word):
    ids = pipe.tokenizer(prompt, truncation=True, max_length=77).input_ids
    w = pipe.tokenizer(word, add_special_tokens=False).input_ids
    for i in range(len(ids) - len(w) + 1):
        if ids[i:i+len(w)] == w: return list(range(i, i+len(w)))
    return None

def attn_loss(maps, masks, token_idx, lam_reg=0.01):
    """Eq. (5). maps: list of [1,N,77]; masks: [1,K,h,w] binary; token_idx: K lists of token ids."""
    tot, n = 0., 0
    for M in maps:
        res = int(math.sqrt(M.shape[1]))
        for k, idx in enumerate(token_idx):
            mk = masks[:, k:k+1]
            m = (mk if mk.shape[-1] == res else (F.interpolate(mk, size=(res, res), mode="area") > 0.3).float()).flatten(1)
            if m.sum() == 0: continue
            a = M[:, :, idx].sum(-1); inside = (a * m).sum(1)
            tot = tot + ((1 - inside / (a.sum(1) + 1e-8)) ** 2 - lam_reg * inside / m.sum(1)).mean(); n += 1
    return tot / n if n else torch.zeros((), device=device)
print("SD2.1 ready; recording cross-attention at", sorted(ATTN_N))

In [ ]:
COCO_NAMES = ["bicycle","car","motorcycle","airplane","bus","train","truck","boat","traffic light","fire hydrant","stop sign",
 "parking meter","bench","bird","cat","dog","horse","sheep","cow","elephant","bear","zebra","giraffe","backpack","umbrella",
 "handbag","suitcase","frisbee","skis","snowboard","sports ball","kite","baseball bat","baseball glove","skateboard","surfboard",
 "tennis racket","bottle","wine glass","cup","fork","knife","spoon","bowl","banana","apple","sandwich","broccoli","carrot","hot dog",
 "pizza","donut","cake","chair","couch","potted plant","bed","dining table","toilet","tv","laptop","keyboard","cell phone",
 "microwave","oven","toaster","sink","refrigerator","book","clock","vase","scissors","teddy bear","toothbrush"]
EXTRA = ["man","woman","men","women","boy","girl","child","kid","kids","people","person","lady","guy","field","garden","beach",
 "street","park","road","tree","trees","building","house","mountain","mountains","lake","river","ocean","sea","sky","grass",
 "forest","hill","snow","kitchen","tower","bridge","water","flower","flowers","plane","jet","table","wave","sand","bush"]
VOCAB = sorted(set(COCO_NAMES) | set(EXTRA), key=lambda w: -len(w))

def extract_objects(caption, max_obj=3):
    c = " " + re.sub(r"[^a-z ]", " ", caption.lower()) + " "; found = []
    for w in VOCAB:
        ww = w if f" {w} " in c else (w + "s" if f" {w}s " in c else None)
        if ww and not any(ww in f or f in ww for f in found): found.append(ww)
    found.sort(key=lambda w: c.find(" " + w))
    return found[:max_obj]

def edge_sketch(img):
    """PIL RGB -> uint8 [H,W] sketch (255 background, 0 strokes)."""
    g = cv2.GaussianBlur(cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2GRAY), (5, 5), 0)
    v = float(np.median(g)); e = cv2.Canny(g, int(max(10, 0.66*v)), int(min(250, 1.33*v + 20)))
    return 255 - cv2.dilate(e, np.ones((2, 2), np.uint8))

def _field(rng, h, w, strength, sigma):
    n = cv2.GaussianBlur(rng.randn(h, w).astype(np.float32), (0, 0), sigma)
    return n / (n.std() + 1e-8) * strength

def augment(sk, mask, rng, freehand):
    """Freehand-style augmentation. sk uint8 [H,W]; mask float [K,r,r] (or None). Mask is warped with the sketch."""
    if not freehand: return sk, mask
    h, w = sk.shape
    dx = _field(rng, h, w, rng.uniform(8, 28), rng.uniform(25, 50)); dy = _field(rng, h, w, rng.uniform(8, 28), rng.uniform(25, 50))
    xx, yy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32)); mx, my = xx + dx, yy + dy
    out = cv2.remap(sk, mx, my, cv2.INTER_LINEAR, borderValue=255); inv = 255 - out
    k = int(rng.choice([1, 2, 3]))
    if k > 1: inv = cv2.dilate(inv, np.ones((k, k), np.uint8))
    p = rng.uniform(0.0, 0.2)                                 
    if p > 0.01:
        blob = cv2.GaussianBlur(rng.rand(h, w).astype(np.float32), (0, 0), 12)
        inv[blob > np.quantile(blob, 1 - p)] = 0
    new_mask = None
    if mask is not None and len(mask):
        r = mask.shape[-1]; ms = []
        for m in mask:
            big = cv2.resize(m.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR)
            big = cv2.remap(big, mx, my, cv2.INTER_LINEAR, borderValue=0)
            ms.append((cv2.resize(big, (r, r), interpolation=cv2.INTER_AREA) > 0.5).astype(np.float32))
        new_mask = np.stack(ms)
    elif mask is not None: new_mask = mask
    return 255 - inv, new_mask


In [ ]:
@torch.no_grad()
def to_latent(img):
    x = torch.from_numpy(np.asarray(img.resize((RES, RES))).copy()).float().div(127.5).sub(1).permute(2, 0, 1)[None]
    return vae.encode(x.to(device, DT)).latent_dist.sample().float() * SCALE

@torch.no_grad()
def attention_masks(z0, prompts, idxs_b, t_probe=250):
    B = z0.shape[0]; emb = torch.cat([get_emb(p) for p in prompts])
    ab = ALPHAS[t_probe]; zt = ab.sqrt() * z0 + (1 - ab).sqrt() * torch.randn_like(z0)
    store.maps, store.active = [], True
    unet(zt.to(DT), torch.full((B,), t_probe, device=device), emb); store.active = False
    agg, cnt = 0, 0
    for M in store.maps:
        r = int(math.sqrt(M.shape[1]))
        if r not in MASK_LAYERS: continue
        agg = agg + F.interpolate(M.view(B, r, r, -1).permute(0, 3, 1, 2), size=(MASK_RES, MASK_RES), mode="bilinear"); cnt += 1
    agg = agg / max(cnt, 1); out = []
    for b in range(B):
        ms = []
        for ix in idxs_b[b]:
            m = F.avg_pool2d(agg[b, ix].sum(0)[None, None], 3, 1, 1)[0, 0]
            m = (m - m.min()) / (m.max() - m.min() + 1e-8); ms.append((m > 0.4).float().cpu().numpy())
        out.append(np.stack(ms) if ms else np.zeros((0, MASK_RES, MASK_RES), np.float32))
    return out

def build_dataset(pairs, bs=8, partial=None):
    data, start = [], 0
    if partial and os.path.exists(partial):
        data, start = pickle.load(open(partial, "rb")); print(f"resuming dataset build at {start} ({len(data)} samples so far)")
    for i in range(start, len(pairs), bs):
        chunk = pairs[i:i+bs]; imgs = [c[0] for c in chunk]; caps = [c[1] for c in chunk]
        z0 = torch.cat([to_latent(im) for im in imgs]); words, idxs = [], []
        for cap in caps:
            w2, i2 = [], []
            for w in extract_objects(cap):
                ix = find_token_idx(cap, w)
                if ix is not None: w2.append(w); i2.append(ix)
            words.append(w2); idxs.append(i2)
        masks = attention_masks(z0, caps, idxs)
        for k in range(len(chunk)):
            keep = [j for j, m in enumerate(masks[k]) if 0.01 < m.mean() < 0.85]
            if keep:
                data.append(dict(caption=caps[k], objs=[words[k][j] for j in keep], idxs=[idxs[k][j] for j in keep],
                                 z0=z0[k:k+1].half().cpu(), sk=edge_sketch(imgs[k].resize((RES, RES))),
                                 mask=masks[k][keep], thumb=imgs[k].resize((256, 256))))
            pairs[i + k] = None                                   # free RAM
        if (i // bs) % 25 == 0: print(f"  processed {i+len(chunk)}/{len(pairs)} -> {len(data)} usable")
        if partial and ((i - start) // bs) % 60 == 59: pickle.dump((data, i + bs), open(partial, "wb"))
    return data

def stream_coco(n):
    from datasets import load_dataset
    cands = [("phiyodr/coco2017", "train"), ("jxie/coco_captions", "train"),
             ("lmms-lab/COCO-Caption2017", "val"), ("nlphuji/mscoco_2014_5k_test_image_text_retrieval", "test")]
    for name, split in cands:
        try:
            got = []; ds = load_dataset(name, split=split, streaming=True)
            for row in ds:
                img = row.get("image") or row.get("img")
                if not isinstance(img, Image.Image): continue
                cap = None
                for key in ("caption", "captions", "answer", "sentences", "text", "raw"):
                    v = row.get(key)
                    if isinstance(v, list) and v: v = v[0]
                    if isinstance(v, dict): v = v.get("raw")
                    if isinstance(v, str) and v.strip(): cap = v.strip(); break
                if not cap or not extract_objects(cap): continue
                img = img.convert("RGB"); s = min(img.size); l, t = (img.width - s)//2, (img.height - s)//2
                got.append((img.crop((l, t, l+s, t+s)).resize((RES, RES), Image.BICUBIC), cap))
                if len(got) >= n: break
            if len(got) >= max(32, n // 4): print(f"using dataset {name}: {len(got)} pairs"); return got
            print(f"{name}: only {len(got)} usable pairs, trying next")
        except Exception as e: print(f"{name} failed: {str(e)[:90]}")
    return []

SUBJ = ["cow","horse","dog","cat","sheep","elephant","giraffe","zebra","bear","bird","man","woman","boy","girl","bus","train",
        "airplane","boat","bicycle","motorcycle","car","bench","tower","clock","kite","umbrella","vase"]
PLACE = ["field","garden","park","beach","street","forest","mountain","lake","snow","city"]
def make_prompt(r):
    a, b, p = r.choice(SUBJ), r.choice(SUBJ), r.choice(PLACE)
    return r.choice([f"a {a} in the {p}", f"a {a} and a {b} in the {p}", f"a photo of a {a} near a {b}, {p}"])

def generate_pairs(n, bs=8, steps=15):
    r = random.Random(0); prompts = [make_prompt(r) for _ in range(n)]; pairs = []; pipe.scheduler = dpm
    for i in range(0, n, bs):
        ps = prompts[i:i+bs]; g = torch.Generator(device).manual_seed(i)
        ims = pipe(ps, height=RES, width=RES, num_inference_steps=steps, generator=g).images
        pairs += list(zip(ims, ps)); print(f"  generated {len(pairs)}/{n}")
    pipe.scheduler = ddim; return pairs

In [ ]:
cache_file = f"{CACHE_DIR}/data_{DATA_SOURCE}_{N_DATA}_{RES}.pkl"
if os.path.exists(cache_file):
    data_all = pickle.load(open(cache_file, "rb")); print("loaded cache:", len(data_all))
else:
    t0 = time.time(); pairs = stream_coco(N_DATA) if DATA_SOURCE == "coco" else []
    if not pairs:
        print("falling back to generated images (slower; lower N_DATA if this takes too long)"); pairs = generate_pairs(min(N_DATA, 200))
    data_all = build_dataset(pairs, partial=cache_file + ".partial"); pickle.dump(data_all, open(cache_file, "wb"))
    if os.path.exists(cache_file + ".partial"): os.remove(cache_file + ".partial")
    print(f"built {len(data_all)} samples in {(time.time()-t0)/60:.1f} min")
for ex in data_all: ex["z0"] = ex["z0"].to(device)
data_test, data_train = data_all[-N_TEST:], data_all[:-N_TEST]
print("train:", len(data_train), "| test:", len(data_test))

rng = np.random.RandomState(0); row = []; ttl = []
for ex in data_train[:3]:
    fsk, fm = augment(ex["sk"], ex["mask"], rng, True)
    row += [ex["thumb"], ex["sk"], fsk, np.kron(fm[0], np.ones((16, 16)))]; ttl += [ex["caption"][:28], "edge sketch", "freehand aug.", f"mask: {ex['objs'][0]}"]
show(row[:6], ttl[:6]); show(row[6:], ttl[6:])

## TRAINING

In [ ]:
def train(data, steps=TRAIN_STEPS, accum=ACCUM, lr=LR, enc_lr=ENC_LR, lam=LAMBDAS, top_frac=TOP_FRAC,
          max_minutes=MAX_MINUTES, resume=True, log_every=20, ckpt_every=100):
    t0 = time.time(); tp = [p for p in enc.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": list(mod.parameters()), "lr": lr}, {"params": tp, "lr": enc_lr}], weight_decay=0.01)
    base, start = [lr, enc_lr], 0
    if resume and os.path.exists(CKPT):
        ck = torch.load(CKPT, map_location=device)
        mod.load_state_dict(ck["mod"]); mod_ema.load_state_dict(ck["ema"]); enc.load_state_dict(ck["enc"], strict=False); start = ck["it"]
        print(f"resumed from step {start}")
    def save(it):
        torch.save({"mod": mod.state_dict(), "ema": mod_ema.state_dict(), "it": it,
                    "enc": {n: p.detach() for n, p in enc.named_parameters() if p.requires_grad}}, CKPT)
    mod.train(); enc.train(); t_lo = int(T_TOTAL * (1 - top_frac)); run = dict(n=0, noise=0., attn=0., reg=0.)
    sd = lambda x: ((x - x.mean()) ** 2).mean().add(1e-8).sqrt()
    for it in range(start, steps):
        f = min(1.0, (it + 1) / 30) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * it / steps)))
        for g, b in zip(opt.param_groups, base): g["lr"] = b * f
        opt.zero_grad(set_to_none=True)
        for _ in range(accum):
            ex = random.choice(data); fh = random.random() < 0.5
            sk_np, mask_np = augment(ex["sk"], ex["mask"], np.random, fh)
            sk = sketch_np_to_tensor(sk_np)[None].to(device); feats = enc(sk)
            masks = torch.from_numpy(mask_np)[None].to(device); emb = get_emb(ex["caption"])
            z0 = ex["z0"].float(); t = torch.randint(t_lo, T_TOTAL, (1,), device=device)
            noise = torch.randn_like(z0); ab = ALPHAS[t].view(1, 1, 1, 1); zt = ab.sqrt() * z0 + (1 - ab).sqrt() * noise
            with torch.no_grad(): eps = unet(zt.to(DT), t, emb).sample.float()
            eps_m, S, B = mod(eps, zt, feats, t)
            l_noise = F.mse_loss(eps_m, noise) if not fh else torch.zeros((), device=device)
            z0_hat = (zt - (1 - ab).sqrt() * eps_m) / ab.sqrt()
            store.maps, store.active = [], True; unet(z0_hat.to(DT), t, emb); store.active = False
            l_attn = attn_loss(store.maps, masks, ex["idxs"]); store.maps = []
            l_reg = S.abs().mean() + B.abs().mean() - (sd(S) + sd(B))
            ((lam[0] * l_noise + lam[1] * l_attn + lam[2] * l_reg) / accum).backward()
            run["n"] += 1; run["noise"] += float(l_noise); run["attn"] += float(l_attn); run["reg"] += float(l_reg)
        torch.nn.utils.clip_grad_norm_(list(mod.parameters()) + tp, 1.0); opt.step()
        d = min(0.99, (1 + it) / (10 + it))
        with torch.no_grad():
            for pe, p in zip(mod_ema.parameters(), mod.parameters()): pe.mul_(d).add_(p.detach(), alpha=1 - d)
        if (it + 1) % log_every == 0:
            n = run["n"]; print(f"step {it+1:4d}/{steps} | noise {run['noise']/n:.4f} | attn {run['attn']/n:.4f} | reg {run['reg']/n:.4f} | {(time.time()-t0)/60:.1f} min")
            run = dict(n=0, noise=0., attn=0., reg=0.)
        if (it + 1) % ckpt_every == 0: save(it + 1)
        if max_minutes and (time.time() - t0) / 60 > max_minutes: print(f"time limit reached at step {it+1}/{steps}. Checkpoint saved -> RE-RUN THIS CELL to continue."); save(it + 1); break
    else: save(steps)
    mod.eval(); enc.eval()

train(data_train, steps=TRAIN_STEPS, max_minutes=MAX_MINUTES)

In [ ]:
from diffusers import ControlNetModel
controlnet = None
try:
    controlnet = ControlNetModel.from_pretrained("thibaud/controlnet-sd21-scribble-diffusers", torch_dtype=DT).to(device).eval()
    print("ControlNet loaded")
except Exception as e: print("ControlNet not available, continuing without it:", str(e)[:100])

In [ ]:
@torch.no_grad()
def generate(prompt, sketch, use_mod=True, steps=30, mod_frac=None, guidance=7.5, seed=0, mod_scale=1.0, cn_scale=0.0):
    """sketch: PIL, dark strokes on white. mod_scale>1 pushes harder on the sketch. cn_scale>0 needs the ControlNet cell."""
    net = mod_ema; enc.eval(); mod_frac = TOP_FRAC if mod_frac is None else mod_frac
    sch = ddim; sch.set_timesteps(steps, device=device)
    emb2 = torch.cat([get_emb(""), get_emb(prompt)])
    lat = torch.randn((1, 4, LAT, LAT), generator=torch.Generator(device).manual_seed(seed), device=device) * sch.init_noise_sigma
    feats = enc(sketch_to_tensor(sketch)[None].to(device))
    if cn_scale > 0 and controlnet is not None:
        hint = torch.from_numpy(255 - np.asarray(sketch.convert("L").resize((RES, RES)))).float().div(255)[None, None].repeat(2, 3, 1, 1).to(device, DT)
    else: cn_scale = 0.0
    for t in sch.timesteps:
        x2 = torch.cat([sch.scale_model_input(lat, t)] * 2).to(DT)
        if cn_scale > 0:
            down, mid = controlnet(x2, t, encoder_hidden_states=emb2, controlnet_cond=hint, conditioning_scale=cn_scale, return_dict=False)
            eps2 = unet(x2, t, encoder_hidden_states=emb2, down_block_additional_residuals=down, mid_block_additional_residual=mid).sample
        else: eps2 = unet(x2, t, encoder_hidden_states=emb2).sample
        eps_u, eps_c = eps2.float().chunk(2)
        if use_mod and int(t) >= int(T_TOTAL * (1 - mod_frac)):
            em, _, _ = net(eps_c, lat, feats, t.view(1)); eps_c = eps_c + mod_scale * (em - eps_c)
        lat = sch.step(eps_u + guidance * (eps_c - eps_u), t, lat).prev_sample
    img = (vae.decode((lat / SCALE).to(DT)).sample.float() / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
    return Image.fromarray((img * 255).round().astype(np.uint8))

def compare(sketch, prompt, seeds=(0, 1), scales=(1.0, 2.5), cn=0.5):
    for sd_ in seeds:
        ims = [sketch, generate(prompt, sketch, use_mod=False, seed=sd_)]; ttl = ["sketch", f"SD2.1 text only (seed {sd_})"]
        for sc in scales: ims.append(generate(prompt, sketch, seed=sd_, mod_scale=sc)); ttl.append(f"ours, mod x{sc}")
        if controlnet is not None and cn > 0:
            ims.append(generate(prompt, sketch, seed=sd_, mod_scale=scales[-1], cn_scale=cn)); ttl.append(f"ours x{scales[-1]} + ControlNet {cn}")
        show(ims, ttl, size=3.3)

compare(demo_sketch, DEMO_PROMPT)

In [ ]:
def adherence(sketch_pil, img):
    x = sketch_to_tensor(img)[None].to(device); s = sketch_to_tensor(sketch_pil)[None].to(device)
    return float((enc.sketch_embed(s) * enc.image_embed(x)).sum())

def evaluate(n=8, steps=20):
    conds = {"text only": dict(use_mod=False), "ours x1": dict(mod_scale=1.0), "ours x2.5": dict(mod_scale=2.5)}
    if controlnet is not None: conds["ours x2.5 + CN0.5"] = dict(mod_scale=2.5, cn_scale=0.5)
    res = {k: [] for k in conds}; gallery = []
    for i, ex in enumerate(data_test[:n]):
        sk_np, _ = augment(ex["sk"], ex["mask"], np.random.RandomState(1000 + i), True); sk = Image.fromarray(sk_np).convert("RGB")
        row = [sk]
        for name, kw in conds.items():
            im = generate(ex["caption"], sk, seed=i, steps=steps, **kw); res[name].append(adherence(sk, im)); row.append(im)
        gallery.append((row, ex["caption"]))
    print("mean sketch-image similarity (higher = better):")
    for k, v in res.items(): print(f"  {k:20s} {np.mean(v):.4f}")
    for row, cap in gallery[:4]: show(row, ["sketch"] + list(conds), size=2.8); print("   caption:", cap)

evaluate()

In [ ]:
from google.colab import files
up = files.upload()                           
my_sketch = Image.open(next(iter(up))).convert("RGB").resize((RES, RES))            
MY_PROMPT = "train"        # <-- edit
compare(my_sketch, MY_PROMPT, seeds=(0, 1, 2), scales=(1.0, 2.5, 4.0))